# Rwanda Landslide Prediction – Data Cleaning & Preprocessing

This notebook explains each code section using comments.

**Workflow:** Load data → inspect → missing values → EDA → outliers → preprocessing → train/test split → save cleaned data.


In [ ]:
# Import pandas for loading and manipulating tabular data
import pandas as pd

# Import matplotlib for creating charts
import matplotlib.pyplot as plt

# Import seaborn for statistical visualizations
import seaborn as sns

# Import ColumnTransformer to apply different preprocessing to different columns
from sklearn.compose import ColumnTransformer

# Import train_test_split to divide data into training and testing sets
from sklearn.model_selection import train_test_split

# Import encoders/scalers for preprocessing
from sklearn.preprocessing import OrdinalEncoder, StandardScaler


In [ ]:
# Store the CSV dataset filename/path
file_path = "Rwanda_Landslide_Dataset.csv"

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first 5 rows to inspect the dataset
display(df.head())


In [ ]:
# Show dataset structure, column names, data types,
# number of non-null values, and memory usage
df.info()


In [ ]:
# Generate descriptive statistics for numerical and categorical columns
# .T transposes the result so features appear as rows
display(df.describe(include="all").T)


In [ ]:
# Count missing values in every column
missing_values = df.isnull().sum()

# Display only columns that contain missing values,
# starting with the columns having the most missing values
display(
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
)

# Calculate the total number of missing cells
print("Total missing values:", int(missing_values.sum()))


In [ ]:
# Create a figure for the missing-value visualization
plt.figure(figsize=(14, 6))

# True values indicate missing cells; False values indicate available cells
sns.heatmap(
    df.isnull(),
    cbar=False,
    yticklabels=False
)

# Add labels and title
plt.title("Heatmap of Missing Values")
plt.xlabel("Features")
plt.ylabel("Observations")

# Rotate feature names so they are easier to read
plt.xticks(rotation=45, ha="right")

# Adjust spacing and display the chart
plt.tight_layout()
plt.show()


In [ ]:
# Count observations for each target class
# 0 = no landslide, 1 = landslide occurred
target_counts = (
    df["landslide_occurred"]
    .value_counts()
    .sort_index()
)

# Create a bar chart of the target classes
plt.figure(figsize=(7, 5))
plt.bar(
    target_counts.index.astype(str),
    target_counts.values
)

# Add chart labels
plt.title("Distribution of Landslide Occurrence")
plt.xlabel("Landslide Occurred")
plt.ylabel("Number of Observations")

# Display the chart
plt.tight_layout()
plt.show()

# Display the exact class counts
display(target_counts.rename("count").to_frame())


In [ ]:
# Select only numerical columns because correlation is calculated for numerical data
numeric_df = df.select_dtypes(include="number")

# Create a larger figure for the correlation matrix
plt.figure(figsize=(14, 10))

# Calculate pairwise correlations and display them as a heatmap
# annot=True writes the correlation values on the cells
# fmt=".2f" displays two decimal places
sns.heatmap(
    numeric_df.corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
# Make a copy so the original dataset remains unchanged
df_cleaned = df.copy()

# Get the names of all numerical columns
num_features = (
    df_cleaned
    .select_dtypes(include="number")
    .columns
    .tolist()
)

# Remove the target because it is what the model will predict
num_features.remove("landslide_occurred")

# Get the names of categorical/text columns
cat_features = (
    df_cleaned
    .select_dtypes(include="object")
    .columns
    .tolist()
)


In [ ]:
# For every numerical feature, replace missing values with the column mean
for column in num_features:
    df_cleaned[column] = (
        df_cleaned[column]
        .fillna(df_cleaned[column].mean())
    )


In [ ]:
# For every categorical feature, replace missing values
# with the most frequent category (mode)
for column in cat_features:
    df_cleaned[column] = (
        df_cleaned[column]
        .fillna(df_cleaned[column].mode()[0])
    )


In [ ]:
# Check for missing values after imputation
remaining_missing = df_cleaned.isnull().sum()

# Show only columns that still contain missing values
display(remaining_missing[remaining_missing > 0])

# Show the total number of missing cells remaining
print(
    "Total missing values after imputation:",
    int(remaining_missing.sum())
)


In [ ]:
# Dictionary used to store the number of IQR outliers per numerical feature
outlier_counts = {}

# Check each numerical feature
for column in num_features:

    # First quartile: 25th percentile
    q1 = df_cleaned[column].quantile(0.25)

    # Third quartile: 75th percentile
    q3 = df_cleaned[column].quantile(0.75)

    # Interquartile Range: Q3 - Q1
    iqr = q3 - q1

    # Values below this limit are potential outliers
    lower_limit = q1 - 1.5 * iqr

    # Values above this limit are potential outliers
    upper_limit = q3 + 1.5 * iqr

    # Count values outside the lower/upper limits
    outlier_counts[column] = int(
        (
            (df_cleaned[column] < lower_limit) |
            (df_cleaned[column] > upper_limit)
        ).sum()
    )


In [ ]:
# Create a table containing outlier counts and non-outlier counts
outlier_report = pd.DataFrame({
    "outlier_count": pd.Series(outlier_counts),
    "non_outlier_count":
        len(df_cleaned) - pd.Series(outlier_counts)
}).sort_values(
    "outlier_count",
    ascending=False
)

# Display the report
display(outlier_report)

# Display the total number of detected IQR outliers
print(
    "Total number of outliers:",
    int(outlier_report["outlier_count"].sum())
)


In [ ]:
# Create box plots for all numerical features
# Box plots help visually identify potential outliers
plt.figure(figsize=(18, 8))

df_cleaned[num_features].boxplot()

plt.title("Box Plot of Numerical Features")
plt.xlabel("Numerical Features")
plt.ylabel("Values")

# Rotate feature names for readability
plt.xticks(rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
# Add a simple status to each numerical feature
# depending on whether its IQR outlier count is greater than zero
display(
    outlier_report.assign(
        status=outlier_report["outlier_count"].apply(
            lambda value:
                "Contains IQR outliers"
                if value > 0
                else "No IQR outliers"
        )
    )
)


In [ ]:
# Re-identify numerical features
num_features = (
    df_cleaned
    .select_dtypes(include="number")
    .columns
    .tolist()
)

# Remove the target from the input features
num_features.remove("landslide_occurred")

# Identify categorical features
cat_features = (
    df_cleaned
    .select_dtypes(include="object")
    .columns
    .tolist()
)

# Print both groups so we can verify the feature types
print("Numerical features:")
print(num_features)

print("\nCategorical features:")
print(cat_features)


In [ ]:
# Create a preprocessing pipeline for different feature types
preprocessor = ColumnTransformer(
    transformers=[

        # Standardize numerical columns
        ("num", StandardScaler(), num_features),

        # Convert categorical values into numeric codes
        # Unknown categories in test data are encoded as -1
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            cat_features
        )
    ]
)

print("Preprocessor created successfully.")


In [ ]:
# X contains all input features
# Remove the target column because the model must predict it
X = df_cleaned.drop(columns=["landslide_occurred"])

# y contains the target variable
y = df_cleaned["landslide_occurred"]

# In this problem:
# 0 = no landslide
# 1 = landslide occurred


In [ ]:
# Split the dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,                       # input features
    y,                       # target
    test_size=0.20,          # 20% for testing, 80% for training
    random_state=42,         # makes the split reproducible
    stratify=y               # preserves target-class proportions
)


In [ ]:
# Display the dimensions of the four split datasets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


In [ ]:
# Show the proportion of each target class in the training set
# normalize=True converts counts into proportions
print("\nTraining target distribution:")

display(
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .rename("proportion")
    .to_frame()
)

# Show the same information for the testing set
print("Testing target distribution:")

display(
    y_test
    .value_counts(normalize=True)
    .sort_index()
    .rename("proportion")
    .to_frame()
)


In [ ]:
# Learn scaling and encoding rules from the training data,
# then transform the training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the already-learned rules to the test data
# We use transform(), not fit_transform(), to avoid data leakage
X_test_processed = preprocessor.transform(X_test)

# Display the dimensions after preprocessing
print("Processed training data shape:", X_train_processed.shape)
print("Processed testing data shape:", X_test_processed.shape)


In [ ]:
# Print a final summary of the cleaned and processed data
print("Cleaned dataset shape:", df_cleaned.shape)

# Count any missing values still remaining
print(
    "Remaining missing values:",
    int(df_cleaned.isnull().sum().sum())
)

# Number of rows in each split
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

# Number of columns after encoding/scaling
print("Training processed columns:", X_train_processed.shape[1])
print("Testing processed columns:", X_test_processed.shape[1])


In [ ]:
# Set the output filename
cleaned_output_path = "Rwanda_Landslide_Dataset_Cleaned.csv"

# Save the cleaned dataset
# index=False prevents pandas from adding an extra index column
df_cleaned.to_csv(
    cleaned_output_path,
    index=False
)

print("Cleaned dataset saved to:", cleaned_output_path)
